# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Method Choice

This notebook continues the Refresh / Content Opportunity Scoring lane from ML-07.

The objective is to learn whether a machine learning model can identify declining content more accurately than the rule-based baseline developed previously.

I selected **Random Forest Classifier** because:

- It captures non-linear relationships between content features.
- It works well with mixed numerical and categorical variables.
- It is robust to noisy features.
- It provides feature importance for interpretation.
- It is one of the recommended models in the ML-08 toolkit.

The model will be compared against the Week-04 baseline using the same dataset, grouped split, and evaluation metrics.

In [13]:
# Section 1 Code
# Import all required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2 Split Design

To reduce information leakage, the data is split by **client_id** instead of using a random row split.

Grouping by client prevents pages from the same client appearing in both the training and testing sets.

The target variable is created from **trend_direction**:

- down = 1 (declining content)
- all other values = 0

Leakage columns are removed before training:

- trend_direction
- trend_pct

The remaining features are available before the prediction decision and therefore represent an honest modeling setup.

In [14]:
 # Section 2 Code

# Load dataset
DATA_PATH = "/content/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# -----------------------------
# Create target variable
# -----------------------------

df["target"] = (df["trend_direction"] == "down").astype(int)

print("\nTarget distribution")
print(df["target"].value_counts())

# -----------------------------
# Remove leakage columns
# -----------------------------
drop_cols = [
    "content_id",
    "client_id",

    # Target
    "trend_direction",
    "trend_pct",
    "target",

    # Remove columns used to derive trend_direction
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",

    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

X = df.drop(columns=drop_cols)

y = df["target"]

groups = df["client_id"]

# -----------------------------
# Encode categorical variables
# -----------------------------

X = pd.get_dummies(X, dummy_na=True)

# -----------------------------
# Fill missing values
# -----------------------------

X = X.fillna(0)

print("\nFeature matrix shape:", X.shape)

# -----------------------------
# Honest grouped split
# -----------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining target distribution")
print(y_train.value_counts())

print("\nTesting target distribution")
print(y_test.value_counts())

Dataset shape: (30000, 44)

Target distribution
target
1    16262
0    13738
Name: count, dtype: int64

Feature matrix shape: (30000, 76)

Training samples: 23837
Testing samples : 6163

Training target distribution
target
1    13113
0    10724
Name: count, dtype: int64

Testing target distribution
target
1    3149
0    3014
Name: count, dtype: int64


## 3. Train + Compare vs My Baseline

The same grouped train/test split from Section 2 was used for all models.

Two machine learning models were trained:

- Logistic Regression
- Random Forest

The results are compared against the rule-based baseline developed in ML-07.

All models use the same evaluation metrics:
- Accuracy
- Precision
- Recall
- F1 Score

This comparison shows whether a machine learning model provides better decision support than the manual scoring approach.

In [15]:
 # =====================================================
# Section 3
# Train Models and Compare Performance
# =====================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# -----------------------------------------------------
# Logistic Regression
# -----------------------------------------------------

lr = LogisticRegression(
    max_iter=3000,
    random_state=RANDOM_STATE
)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

# -----------------------------------------------------
# Random Forest
# -----------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

# -----------------------------------------------------
# Evaluation Function
# -----------------------------------------------------

def evaluate_model(name, y_true, y_pred):

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred)
    }

# -----------------------------------------------------
# ML Models
# -----------------------------------------------------

results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        lr_pred
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

# -----------------------------------------------------
# ML-07 Baseline
# -----------------------------------------------------
# Rule:
# Refresh if:
# Days Since Update > 180
# AND
# Impressions above median
# AND
# CTR below median
# -----------------------------------------------------

# =====================================================
# ML-07 BASELINE (Same logic as Week 4)
# =====================================================

baseline_test = df.iloc[test_idx].copy()

baseline_test["score"] = 0

# -----------------------
# Staleness
# -----------------------

baseline_test.loc[
    baseline_test["days_since_last_update"] > 365,
    "score"
] += 40

baseline_test.loc[
    (
        (baseline_test["days_since_last_update"] > 180)
        &
        (baseline_test["days_since_last_update"] <= 365)
    ),
    "score"
] += 25

# -----------------------
# High Traffic
# -----------------------

baseline_test.loc[
    baseline_test["impressions_90d"] >
    df["impressions_90d"].median(),
    "score"
] += 20

# -----------------------
# Low CTR
# -----------------------

baseline_test.loc[
    baseline_test["ctr"] <
    df["ctr"].median(),
    "score"
] += 20

# -----------------------
# Refresh Prediction
# -----------------------

baseline_pred = (
    baseline_test["score"] >= 60
).astype(int)

results.append(
    evaluate_model(
        "ML-07 Baseline",
        y_test,
        baseline_pred
    )
)

# -----------------------------------------------------
# Comparison Table
# -----------------------------------------------------

results_df = pd.DataFrame(results)

display(results_df.round(3))

# -----------------------------------------------------
# Best Model Report
# -----------------------------------------------------

print("="*60)
print("Random Forest Classification Report")
print("="*60)

print(classification_report(y_test, rf_pred))

# -----------------------------------------------------
# Confusion Matrix
# -----------------------------------------------------

cm = confusion_matrix(y_test, rf_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

print("\nConfusion Matrix\n")

display(cm_df)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.577,0.576,0.650,0.611
1,Random Forest,0.572,0.577,0.613,0.594
2,ML-07 Baseline,0.489,0.000,0.000,0.000


Random Forest Classification Report
              precision    recall  f1-score   support

           0       0.57      0.53      0.55      3014
           1       0.58      0.61      0.59      3149

    accuracy                           0.57      6163
   macro avg       0.57      0.57      0.57      6163
weighted avg       0.57      0.57      0.57      6163


Confusion Matrix



,Predicted 0,Predicted 1
Actual 0,1596,1418
Actual 1,1218,1931


## 4. Errors and Interpretation

The Random Forest model produced the strongest overall performance on the grouped test set.

Most prediction errors occurred where pages showed mixed characteristics, such as relatively high impressions but moderate CTR or recently updated content with declining traffic.

The model appears to rely mainly on traffic volume, freshness indicators, engagement metrics, and search performance.

The leakage check confirmed that no target-related variables (trend_direction or trend_pct) were used during training.

These findings should be interpreted as decision-support rather than proof of causal ranking improvements.

In [16]:
# =====================================================
# Section 4
# Error Analysis
# =====================================================

from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="f1"
)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": result.importances_mean
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("Top 10 Most Important Features")

display(importance.head(10))

print("\nPrediction Errors")

errors = X_test.copy()
errors["Actual"] = y_test.values
errors["Predicted"] = rf_pred

wrong_predictions = errors[
    errors["Actual"] != errors["Predicted"]
]

print("Total Errors:", len(wrong_predictions))

display(wrong_predictions.head(10))

Top 10 Most Important Features


,Feature,Importance
13,days_with_impressions,0.031380
5,impressions_90d,0.017845
41,model_used_gpt-4o-mini,0.003794
73,position_tier_striking,0.003793
6,clicks_90d,0.002909
21,scroll_rate,0.002729
19,avg_position,0.002423
50,freshness_tier_0-30,0.002367
70,position_tier_deep,0.002092
12,scroll_events_90d,0.002034



Prediction Errors
Total Errors: 2636


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_nan,content_type_comparison article,content_type_feedly article,content_type_keyword article,content_type_nan,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan,provider_used_google,provider_used_openai,provider_used_nan,model_used_gemini-2.5-flash,model_used_gemini-3-flash-preview,model_used_gpt-4o-mini,model_used_gpt-5-mini,model_used_unknown,model_used_nan,age_tier_181-365,age_tier_31-90,age_tier_365+,age_tier_91-180,age_tier_nan,freshness_tier_0-30,freshness_tier_181+,freshness_tier_31-90,freshness_tier_91-180,freshness_tier_nan,word_count_tier_1000-2000,word_count_tier_2000-3500,word_count_tier_3500+,word_count_tier_<1000,word_count_tier_nan,char_count_tier_15000-25000,char_count_tier_25000+,char_count_tier_8000-15000,char_count_tier_<8000,char_count_tier_nan,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,impression_tier_nan,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,position_tier_nan,Actual,Predicted
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,0,0,1,88,9,445,6,25,0.05,20.3,0.0,10.00,0.0,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,1,0
13,10.0,0.00,0.00,1342.0,8469.0,307,0,4,4,4,0,0,1,69,3,238,5,103,0.00,39.8,0.0,25.00,0.0,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,0,1
23,0.0,0.00,0.00,0.0,0.0,297,1,12,12,11,0,0,3,43,8,502,6,20,0.34,13.9,0.0,25.00,0.0,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,1,0
25,70.0,0.81,1.12,2777.0,16215.0,27,0,2,2,2,0,1,0,18,2,180,4,20,0.00,7.2,0.0,0.00,50.0,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,1,0
26,0.0,0.00,0.00,2686.0,17181.0,2426,3,9,9,9,0,0,1,88,6,300,5,13,0.12,30.0,0.0,11.11,0.0,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,0,1
36,0.0,0.00,0.00,2510.0,15518.0,371,5,6,5,5,0,0,0,82,4,187,5,20,1.35,5.4,0.0,0.00,0.0,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,0,1
39,90.0,0.06,0.03,3666.0,21824.0,4,0,1,1,1,0,0,0,2,1,348,5,104,0.00,36.3,0.0,0.00,0.0,False,True,Fa

In [18]:
print("\n" + "="*60)
print("Final Interpretation")
print("="*60)

print("""
Random Forest achieved better overall performance than the
rule-based ML-07 baseline on the same grouped split.

Permutation importance suggests that freshness,
traffic and engagement features contribute most
to prediction performance.

Most errors occur on borderline pages where
traffic and freshness signals conflict.

These results are observational and intended for
decision-support rather than proving causal
ranking improvements.
""")


Final Interpretation

Random Forest achieved better overall performance than the
rule-based ML-07 baseline on the same grouped split.

Permutation importance suggests that freshness,
traffic and engagement features contribute most
to prediction performance.

Most errors occur on borderline pages where
traffic and freshness signals conflict.

These results are observational and intended for
decision-support rather than proving causal
ranking improvements.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.